# Generic Tuning Interface

In [1]:
from arbok_driver import ArbokDriver, Device, Measurement
from arbok_driver.examples.configurations import opx1000_config

2026-03-20 05:21:16,188 - qm - INFO     - Starting session: bebcaf40-f2af-4467-be5b-1324e45d7c56


In [2]:
mock_device = Device(
    name = 'mock_device',
    opx_config = opx1000_config,
    divider_config = {})

qm_driver = ArbokDriver('qm_driver', mock_device)

In [3]:
from arbok_driver.examples.sub_sequences import StabilityMap
from arbok_driver.examples.configurations import stability_map_config

mock_measurement = Measurement(qm_driver, 'mock_measurement')
stability_map = StabilityMap(
    mock_measurement, 'stability_map', stability_map_config)

In [4]:
from arbok_driver.generic_tunig_interface import CostStrategy

import numpy as np

class StabStrat(CostStrategy):
    def get_cost(self, results: dict) -> float:
        return float(np.random.rand())
    
stability_cost_strat = StabStrat(
    mock_measurement.available_gettables,
    [{stability_map.iteration: np.arange(100)}]
    )

In [5]:
[g.full_name for g in mock_measurement.available_gettables]

['qm_driver_mock_measurement_stability_map_p1p2_read',
 'qm_driver_mock_measurement_stability_map_p1p2_ref',
 'qm_driver_mock_measurement_stability_map_p1p2_diff',
 'qm_driver_mock_measurement_stability_map_p7p8_read',
 'qm_driver_mock_measurement_stability_map_p7p8_ref',
 'qm_driver_mock_measurement_stability_map_p7p8_diff']

In [6]:
mock_measurement.print_readable_snapshot()

qm_driver_mock_measurement:
	parameter value
--------------------------------------------------------------------------------
qm_driver_mock_measurement_stability_map:
	parameter                 value
--------------------------------------------------------------------------------
chop__stability__chop_wait :	5000 (s)
chop__stability__n_chops   :	20 
chop__stability__v_chop_P1 :	0.01 (V)
chop__stability__v_chop_P2 :	-0.01 (V)
chop__stability__v_chop_P7 :	0.01 (V)
chop__stability__v_chop_P8 :	-0.01 (V)
gate_elements              :	['P1', 'J1', 'P2', 'J2', 'P3', 'J3', 'P4', 'J4',...
iteration                  :	0 
p1p2_diff                  :	Not available 
p1p2_read                  :	Not available 
p1p2_ref                   :	Not available 
p7p8_diff                  :	Not available 
p7p8_read                  :	Not available 
p7p8_ref                   :	Not available 
t_pre_chop                 :	12500 (s)
v_home_J1                  :	0 (V)
v_home_J2                  :	0 (V)
v_home_

In [7]:
mock_measurement.initialize_tuning_interface(
    parameter_dicts = {
        'test1': {
            'qua_vars': {stability_map.arbok_params.v_home['J1']: 1},
            'bounds': (-0.1, 0.1)
            },
        'test2': {
            'qua_vars': {
                stability_map.arbok_params.v_home['J2']: 1,
                stability_map.arbok_params.v_home['P2']: -1,
                },
            'bounds': (-0.01, 0.01)
            },
        },
    cost_strategy = stability_cost_strat,
    verbose = True
)

Adding input stream for v_home_J1 (test1)

test1

Adding input stream for v_home_J2 (test2)

Adding v_home_P2 to v_home_J2  input stream (factor: -1) (test2)

test2

Registered 6 gettables for measurement
Declared 1-dimensional parameter sweep of size 100 [100]


In [8]:
mock_measurement.print_qua_program_to_file('tet.py')